In [1]:
%%capture
!pip install facenet-pytorch

In [2]:
import sys
sys.path.append('/home/pj00/projects/Github/small_face_recognition_trcking/utils')

## Libraries

In [13]:
from facenet_pytorch import MTCNN, InceptionResnetV1, fixed_image_standardization, training

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models
from torchvision import transforms
from torchsummary import summary

from PIL import Image

import numpy as np
np.bool = np.bool_

import mxnet as mx
from mxnet import recordio

from image_iter import FaceDataset
from custom_model import distill_model
from utils import model_size

from tqdm import tqdm

## Functions

## Models

In [4]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [8]:
# Load model
'''
The cropped faces are passed as input in the CNN and we get an embedding for each face.
Important to set the model at .eval()
'''
teacher = InceptionResnetV1(
    classify=True,
    pretrained='casia-webface').to(device)

num_classes = teacher.logits.out_features
model_size(teacher)

model size: 925043168 / bit | 115.63 / MB


In [9]:
student = distill_model(num_classes)
student.to(device)
model_size(student)

model size: 395431392 / bit | 49.43 / MB


## Load Data

In [10]:
BATCH_SIZE=32
train_root = '/home/pj00/projects/Github/small_face_recognition_trcking/Data/CASIA/casia-webface/train.rec'

In [11]:
dataset = FaceDataset(path_imgrec=train_root, rand_mirror=True)
train_loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

/home/pj00/projects/Github/small_face_recognition_trcking/Data/CASIA/casia-webface/train.rec /home/pj00/projects/Github/small_face_recognition_trcking/Data/CASIA/casia-webface/train.idx
header0 label [490624. 501196.]
id2range 10572


## Training

In [15]:
criterion = nn.MSELoss()
optimizer = optim.Adam(student.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5, verbose=True)

/home/pj00/anaconda3/envs/CVenv/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


In [16]:
x, y = next(iter(train_loader))

In [18]:
teacher.eval()
student.train()
'Done'

'Done'

In [20]:
with torch.no_grad():
    y1 = teacher(x.to(device, dtype=torch.float32))
y2 = student(x.to(device, dtype=torch.float32))

In [21]:
loss = criterion(y1, y2)
optimizer.zero_grad()
loss.backward()
optimizer.step()

In [22]:
loss

tensor(4614.2456, device='cuda:0', grad_fn=<MseLossBackward0>)

In [30]:
teacher.eval()
student.train()

losses = []
epochs=100

for epoch in tqdm(range(epochs)):
    epoch_loss = 0.0
    for images, _ in train_loader, leave=True:
        images = images.to(device, dtype=torch.float32)
        
        with torch.no_grad():
            embed_teacher = teacher(x.to(device, dtype=torch.float32))
        embed_student = student(x.to(device, dtype=torch.float32))
        
        loss = criterion(embed_teacher, embed_student)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    epoch_loss /= len(train_loader)
    losses.append(epoch_loss)
    print('Epoch_loss: {}'.fomrat(epoch_loss))
    
    scheduler.step(epoch_loss)

  0%|                                                   | 0/3 [00:03<?, ?it/s]


KeyboardInterrupt: 